In [6]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

#setup project root and paths
project_root = Path("/dcs/23/u2200504/thesis/recidivism-causal").resolve()

#set causal pitfalls root
cp_root = project_root / "data" / "raw" / "CausalPitfallsData"

#path to put results
output_dir = project_root/"results"/"graphs_DAGSLAM_scored"
output_dir.mkdir(parents=True,exist_ok=True)

#test it works
print("CP roots:" , cp_root)
print("Output directory:", output_dir)

CP roots: /dcs/23/u2200504/thesis/recidivism-causal/data/raw/CausalPitfallsData
Output directory: /dcs/23/u2200504/thesis/recidivism-causal/results/graphs_DAGSLAM_scored


In [3]:
#add dagslam implementation to system path
dagslam_path = project_root/"code"/"dagslam"/"DAGSLAM Causal Bayesian Network Structure Learning of Mixed Type Data"/"_dagslam"
sys.path.append(str(dagslam_path))

#uses DAGSLAM implementation code authored by Yuanyuan Zhao. 
#https://github.com/yuanyuan-zhao-pku/DAGSLAM/blob/main/DAGSLAM%20Causal%20Bayesian%20Network%20Structure%20Learning%20of%20Mixed%20Type%20Data/_dagslam/DAGSLAM.py
#DAGSLAM is an extension of the NOTEARS algorithm developed by Xun Zheng, et al.
import importlib
import DAGSLAM 
importlib.reload(DAGSLAM)

from DAGSLAM import dagslam

#m_vec gives the total number of categories for each multinomial variable
#helper function to infer loss types and generate m_vec for each column
def infer_type(df):
    loss_type=[]
    m_vec=[]

    for col in df.columns:
        x=df[col].dropna()
        unique_vals=x.unique()
        num_unique = len(unique_vals)

        #infer variable type by combination of dtype and cardinality
        if np.issubdtype(x.dtype, np.number):
            #numeric datatypes
            if num_unique == 2 and set(unique_vals).issubset({0,1}): #binary numeric variable
                loss_type.append("logistic")
                m_vec.append(1) # 1 "category"
            else: #continuous 
                loss_type.append("gauss")
                m_vec.append(1)
        else: #non-numeric
            if num_unique == 2:
                #binary categorical 
                loss_type.append("logistic")
                m_vec.append(1)
            else:
                #multi-class categorical
                loss_type.append("multi-logistic")
                m_vec.append(num_unique)

    return loss_type, m_vec

def run_dagslam(df, lambda1=0.03, max_iter=100, w_threshold=0.25): #changed lambda 1 0.03 ->0.1 w_threshold 0.25->0.3
    df_clean = df.dropna().copy()
    loss_type, m_vec =infer_type(df_clean)
    X=df_clean.values
    
    W_est = dagslam(X, loss_type=loss_type, m_vec=m_vec, lambda1=lambda1,max_iter=max_iter, w_threshold=w_threshold)
    return W_est

#draw graphs based on DAGSLAM weighted adjacency matrix
def draw_graph(W, output_path, node_labels=None,threshold=0):
    n = W.shape[0] # number of nodes
    G = nx.DiGraph() # initialise empty directed graph
    
    #initialise default node labels
    if node_labels is None:
        node_labels = [f"X{i}" for i in range(n)]

    #add nodes to digraph
    for i, name in enumerate(node_labels):
        G.add_node(i, label=name)

    #add edges for surviving weights (thresholding done during the dagslam phase)
    for i in range(n):
        for j in range(n): # for each possible edge 
            w=W[i,j]
            if abs(w)>threshold:
                G.add_edge(i, j, weight=w)

    plt.figure(figsize=(6,6))
    pos = nx.spring_layout(G, seed=0)
    nx.draw(G, pos, with_labels=True, labels={i: node_labels[i] for i in range(n)},
            node_size=800, font_size=8, arrowsize=10)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()

In [24]:
from graphviz import Digraph
from pathlib import Path
import numpy as np

def draw_graphviz_dag(W, out_path, node_labels=None, threshold=0.0, engine="dot"):

    W = np.asarray(W)
    if W.ndim == 1:
        W = W.reshape(1, 1)

    n = W.shape[0]

    # Default labels use X0, X1,...
    if node_labels is None:
        node_labels = [f"X{i}" for i in range(n)]
    else:
        node_labels = list(node_labels)[:n]

    out_path = Path(out_path)
    g = Digraph(format="svg", engine=engine)

    g.attr(rankdir="LR")
    g.attr(
        "node",
        shape="ellipse",
        style="solid",
        color="black",
        fontname="Helvetica",   # or "Times New Roman" / "Palatino"
        fontsize="10",
    )
    g.attr(
        "edge",
        color="black",
        arrowsize="0.7",
    )

    # Add nodes
    for name in node_labels:
        g.node(name, label=name)

    # Add edges for weights above threshold
    for i, src in enumerate(node_labels):
        for j, tgt in enumerate(node_labels):
            w = W[i, j]
            if abs(w) > threshold:
                # You can optionally add weight as label=str(round(w, 2))
                g.edge(src, tgt)

    out_path.parent.mkdir(parents=True, exist_ok=True)
    g.render(filename=out_path.with_suffix("").as_posix(), cleanup=True)


In [25]:
#scoring utility function given two adjacency matricies
def score_graph(W_est, W_true, labels_est, labels_true):
    #number of variables
    p = W_est.shape[0]

    #node orderings
    labels_est = list(labels_est)
    labels_true = list(labels_true)
    common = sorted(set(labels_est) & set(labels_true))

    #map ids
    idx_est = [labels_est.index(v) for v in common]
    idx_true = [labels_true.index(v) for v in common]

    W_est_aligned = np.asarray(W_est)[np.ix_(idx_est, idx_est)]
    W_true_aligned = np.asarray(W_true)[np.ix_(idx_true, idx_true)]

    est = (W_est!=0).astype(int)
    true = (W_true !=0).astype(int)

    #true positive, false positive, false negative, true negative
    tp = np.sum((est==1) & (true==1))
    fp = np.sum((est==1) & (true == 0))
    fn = np.sum((est==0) & (true == 1))
    tn = np.sum((est==0) & (true == 0))

    #structural hamming distance, true positive rate, false positive rate
    shd = fp + fn
    tpr = tp/(tp+fn) if (tp+fn) > 0 else np.nan
    fpr = fp/(fp+tn) if (fp + tn)>0 else np.nan
    fdr = fp/(tp+fp) if (tp+fp)>0 else np.nan

    #store scores in dictionary format for each 'experiment'
    return dict(TP=int(tp), FP=int(fp), FN=int(fn), TN=int(tn), SHD=int(shd), TPR = tpr, FPR = fpr, FDR= fdr)

In [26]:
#recurse over all csv datasets and run DAGSLAM
csv_files = sorted(cp_root.rglob("*.csv"))
print(f"Found {len(csv_files)} CSV files")
results = []

for csv_path in csv_files:
    rel = csv_path.relative_to(cp_root)

    #we only examine those with known ground truths i.e with _truth
    if csv_path.stem.endswith("_truth"):
        continue

    #expected ground truth path
    truth_path = csv_path.with_name(csv_path.stem + "_truth.csv")
    if not truth_path.exists():
        print("  Skipped (no ground-truth file)")
        continue

    print(f"Processing: {rel}")

    try:
        df = pd.read_csv(csv_path)
        if df.empty:
            print("  Skipped (empty file)")
            continue

        # Run DAGSLAM on this dataset
        W_est = run_dagslam(df)

        #Output schema scenario__file__dagslam.png
        parts = rel.parts           
        scenario = parts[0] if len(parts) > 1 else "root"
        name_no_ext = csv_path.stem

        out_name = f"{scenario}__{name_no_ext}__dagslam.png"
        out_path = output_dir / out_name

        # Save PNG
        #draw_graph(W_est, out_path, node_labels=list(df.columns))

        #use graphviz instead
        draw_graphviz_dag(W_est, out_path, node_labels=list(df.columns), threshold=0.0)
        print(f"  Saved graph to {out_path.relative_to(project_root)}")

        #load ground truth and score
        gt_df = pd.read_csv(truth_path)
        gt_df.index.name = None
        gt_df.columns.name = None

        W_true = gt_df.to_numpy()
            
        # compute scores
        metrics = score_graph(W_est, W_true, labels_est=list(df.columns), labels_true=list(gt_df.columns))
        print("metrics computed")
        print(metrics)
        metrics.update(
            dict(
                scenario=scenario,
                dataset=name_no_ext,
                algo="dagslam"
            )
        )
        results.append(metrics)
        print(f"  SHD={metrics['SHD']}, TPR={metrics['TPR']:.3f}, FDR={metrics['FDR']:.3f}")

    except Exception as e:
        print(f"  ERROR on {rel}: {e}")

scores_df = pd.DataFrame(results)

Found 171 CSV files
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
  Skipped (no ground-truth file)
Processing: casual_effect/device_failure_data.csv
iter:0
rho:1.0
loss=1007267.30610992
loss=1007267.30610992
loss=1007267.30610992
loss=1007267.30610992
loss=1007267.30610992
loss=1007267.30610992
loss=1007267.30610992
loss=1007267.30610992
loss=1007267.30610992
loss=1007267.30610992
loss=1007267.30610992
loss=1007267.30610992
loss=1007267.30610992
loss=1007267.30610992
loss=1007267.30610992
loss=1007267.30610992
loss=1007267.30610992
loss=1007267.30610992
loss=1007267.30610992
lo

/dcs/23/u2200504/thesis/envthesis/lib/python3.13/site-packages/scipy/linalg/_matfuncs.py:389: RuntimeWarning: overflow encountered in matmul
  eAw = eAw @ eAw
/dcs/23/u2200504/thesis/envthesis/lib/python3.13/site-packages/scipy/linalg/_matfuncs.py:389: RuntimeWarning: invalid value encountered in matmul
  eAw = eAw @ eAw


loss=1.5793732730277777e+20
loss=1.7408627974128455e+20
loss=2.0543360618091446e+20
loss=2.2846696921316426e+20
loss=4.12688399807212e+20
loss=2.4880309618497952e+20
loss=2.9163368250712896e+20
loss=3.44220162153845e+20
loss=3.469922905730881e+20
loss=1.5918465734106684e+24
loss=3.46326124037365e+20
loss=3.462069782032101e+20
loss=3.457319786954845e+20
loss=3.438573219268837e+20
loss=3.367641550493065e+20
loss=2.9381494124384164e+20
loss=2.799420872810209e+20
loss=1.9859417554907588e+20
loss=1.918585797957498e+20
loss=1.906171360145516e+20
loss=1.8994875164365265e+20
loss=1.8763798586047522e+20
loss=1.76740284985435e+20
loss=1.6316232467986206e+20
loss=1.572776959685295e+20
loss=1.5645800682420332e+20
loss=1.5499315153596016e+20
loss=1.4423351779382716e+20
loss=1.335583512771895e+20
loss=1.3005187332839021e+20
loss=1.2706927824635239e+20
loss=1.2687898666606146e+20
loss=1.2644280034476193e+20
loss=1.261909952088933e+20
loss=1.260514475659191e+20
loss=1.2666422387160901e+20
loss=1.28675

/dcs/23/u2200504/thesis/recidivism-causal/code/dagslam/DAGSLAM Causal Bayesian Network Structure Learning of Mixed Type Data/_dagslam/DAGSLAM.py:104: RuntimeWarning: invalid value encountered in logaddexp
  * (np.logaddexp(0, M[..., j]) - X[..., j] * M[..., j])
/dcs/23/u2200504/thesis/envthesis/lib/python3.13/site-packages/numpy/linalg/linalg.py:2120: RuntimeWarning: invalid value encountered in slogdet
  sign, logdet = _umath_linalg.slogdet(a, signature=signature)


loss=584.0097065274559
loss=584.0095558333279
loss=584.008791887675
loss=584.0087414698055
loss=584.008497816654
loss=584.0082515038492
loss=584.0079635348511
loss=584.0078121889374
loss=584.0074555933846
loss=584.007688413449
loss=584.0078032050444
loss=584.007847104019
loss=584.0078628506478
loss=584.007876448669
loss=584.0078532681534
loss=584.959991402057
loss=584.0078294188544
loss=584.0075693377928
loss=584.0071124888232
loss=584.0119581775326
loss=584.0071723199163
loss=584.0067312837384
loss=584.0065951177253
loss=584.0065588286086
loss=584.0065761175621
loss=584.0066347128776
loss=584.0066760943241
loss=584.0070154224321
loss=584.0075031829094
loss=584.0066102740668
loss=584.007308494477
loss=584.0080101661373
loss=584.008028617389
loss=584.0158047442179
loss=584.0081764286348
loss=584.007351796043
loss=584.006473297316
loss=584.0052052304966
loss=584.0056696446978
loss=584.0056528212227
loss=584.00564747341
loss=584.0056842123919
loss=584.0058184825392
loss=584.0061495308316


...

loss=1.2640148565884405e+17
loss=1.2629037163144083e+17
loss=1.26198915082286e+17
loss=1.2607779139990682e+17
loss=1.259902203206741e+17
loss=1.2588796168966107e+17
loss=1.2574917016459467e+17
loss=1.25859245393627e+17
loss=1.2606174524214477e+17
loss=1.2635545684926686e+17
loss=1.265270900160773e+17
loss=1.2652335653284238e+17
loss=1.2642651222407616e+17
loss=1.2641571161552006e+17
loss=1.2644948893418986e+17
loss=1.2641881005603579e+17
loss=1.2647986700711264e+17
loss=1.265642557297513e+17
loss=1.2683581539381069e+17
loss=1.2705990180900056e+17
loss=1.2721335375899643e+17
loss=1.2764724852608478e+17
loss=1.2740533726678558e+17
loss=1.275862454687333e+17
loss=1.2760932429132114e+17
loss=1.2763801740104642e+17
loss=1.2774416559335976e+17
loss=1.2775324177129797e+17
loss=1.2780839390549331e+17
loss=1.2786658040869107e+17
loss=1.2809767683198136e+17
loss=1.2836870945221941e+17
loss=1.2831326559162622e+17
loss=1.2822343804088606e+17
loss=1.2812308114186845e+17
loss=1.2796143512828986e+17


/dcs/23/u2200504/thesis/recidivism-causal/code/dagslam/DAGSLAM Causal Bayesian Network Structure Learning of Mixed Type Data/_dagslam/DAGSLAM.py:104: RuntimeWarning: invalid value encountered in logaddexp
  * (np.logaddexp(0, M[..., j]) - X[..., j] * M[..., j])
/dcs/23/u2200504/thesis/envthesis/lib/python3.13/site-packages/numpy/linalg/linalg.py:2120: RuntimeWarning: invalid value encountered in slogdet
  sign, logdet = _umath_linalg.slogdet(a, signature=signature)


loss=2.3326164172764487e+83
loss=2.346058821783571e+83
loss=2.3548267678038477e+83
loss=2.3593888022182244e+83
loss=2.354999988373488e+83
loss=2.339954296149285e+83
loss=2.3321743218049392e+83
loss=2.3278059419828532e+83
loss=2.3330777689491166e+83
loss=2.349189802025418e+83
loss=2.3974916675158462e+83
loss=2.4045561366572728e+83
loss=2.4084582461297624e+83
loss=2.4090309989535301e+83
loss=2.4092622227623055e+83
loss=2.4092127111779858e+83
loss=2.4102196505976594e+83
loss=2.4075898731471704e+83
loss=2.4062561160420302e+83
loss=2.4044735275986565e+83
loss=2.4007566013288406e+83
loss=2.3995489010613284e+83
loss=2.3994972312715192e+83
loss=2.3989621091538253e+83
loss=2.4000943878570224e+83
loss=2.4021070844064613e+83
loss=2.4048025025577638e+83
loss=2.4046990347216513e+83
loss=2.401115402580709e+83
loss=2.3940686946736455e+83
loss=2.3814514420852614e+83
loss=2.3729205518760718e+83
loss=2.364109441201475e+83
loss=2.3672006025216295e+83
loss=2.3765069884877222e+83
loss=2.3831678063652357e+8

/dcs/23/u2200504/thesis/recidivism-causal/code/dagslam/DAGSLAM Causal Bayesian Network Structure Learning of Mixed Type Data/_dagslam/DAGSLAM.py:104: RuntimeWarning: invalid value encountered in logaddexp
  * (np.logaddexp(0, M[..., j]) - X[..., j] * M[..., j])
/dcs/23/u2200504/thesis/envthesis/lib/python3.13/site-packages/numpy/linalg/linalg.py:2120: RuntimeWarning: invalid value encountered in slogdet
  sign, logdet = _umath_linalg.slogdet(a, signature=signature)
/dcs/23/u2200504/thesis/recidivism-causal/code/dagslam/DAGSLAM Causal Bayesian Network Structure Learning of Mixed Type Data/_dagslam/DAGSLAM.py:104: RuntimeWarning: invalid value encountered in logaddexp
  * (np.logaddexp(0, M[..., j]) - X[..., j] * M[..., j])
/dcs/23/u2200504/thesis/envthesis/lib/python3.13/site-packages/numpy/linalg/linalg.py:2120: RuntimeWarning: invalid value encountered in slogdet
  sign, logdet = _umath_linalg.slogdet(a, signature=signature)


,TP,FP,FN,TN,SHD,TPR,FPR,FDR,scenario,dataset,algo
0,2,11,7,29,18,0.222222,0.275000,0.846154,casual_effect,device_failure_data,dagslam
1,9,26,0,14,26,1.000000,0.650000,0.742857,casual_effect,student_tutoring_data,dagslam
2,3,1,0,12,1,1.000000,0.076923,0.250000,causal_direction_iv,clinical_trial_sem,dagslam
3,2,7,1,6,8,0.666667,0.538462,0.777778,causal_direction_iv,ecommerce_sem,dagslam
4,2,7,1,6,8,0.666667,0.538462,0.777778,causal_direction_iv,environment_sem,dagslam
5,3,6,0,7,6,1.000000,0.461538,0.666667,causal_direction_iv,marketing_sem,dagslam
6,0,3,4,9,7,0.000000,0.250000,1.000000,counterfactual_reasoning,climate_impact_sem,dagslam
7,0,3,4,9,7,0.000000,0.250000,1.000000,counterfactual_reasoning,clinical_trial_sem,dagslam
8,0,3,4,9,7,0.000000,0.250000,1.000000,counterfactual_reasoning,education_performance_sem,dagslam
9,0,3,4,9,7,0.000000,0.250000,1.000000,counterfactual_reasoning,investment_outcome_sem,dagslam


In [14]:
#encountered overflows in matrix multiplication

In [27]:
scores_df

,TP,FP,FN,TN,SHD,TPR,FPR,FDR,scenario,dataset,algo
0,2,11,7,29,18,0.222222,0.275000,0.846154,casual_effect,device_failure_data,dagslam
1,9,26,0,14,26,1.000000,0.650000,0.742857,casual_effect,student_tutoring_data,dagslam
2,3,1,0,12,1,1.000000,0.076923,0.250000,causal_direction_iv,clinical_trial_sem,dagslam
3,2,7,1,6,8,0.666667,0.538462,0.777778,causal_direction_iv,ecommerce_sem,dagslam
4,2,7,1,6,8,0.666667,0.538462,0.777778,causal_direction_iv,environment_sem,dagslam
5,3,6,0,7,6,1.000000,0.461538,0.666667,causal_direction_iv,marketing_sem,dagslam
6,0,3,4,9,7,0.000000,0.250000,1.000000,counterfactual_reasoning,climate_impact_sem,dagslam
7,0,3,4,9,7,0.000000,0.250000,1.000000,counterfactual_reasoning,clinical_trial_sem,dagslam
8,0,3,4,9,7,0.000000,0.250000,1.000000,counterfactual_reasoning,education_performance_sem,dagslam
9,0,3,4,9,7,0.000000,0.250000,1.000000,counterfactual_reasoning,investment_outcome_sem,dagslam


In [28]:
scores_path = project_root / "results" / "scores"/"scores_dagslam.csv"
scores_path.parent.mkdir(parents=True, exist_ok=True)
scores_df.to_csv(scores_path, index=False)